# RAG Databricks Bluetab - Embedding Model Registration

## Overview
This notebook registers a custom embedding model for the RAG pipeline using MLflow and Databricks Model Registry.

## Features
- Custom embedding model using DistilBERT
- MLflow model packaging with proper dependencies
- Databricks Vector Search compatibility
- Parameterized model registration
- Comprehensive logging and monitoring

## Model Details
- Base Model: `distilbert-base-uncased`
- Framework: PyTorch + Transformers
- Output: 768-dimensional embeddings
- Inference: Batch processing capable

## Dependencies
- Run `00 Configuration and Utils` notebook first

In [0]:
%run "./00 Configuration and Utils"

In [0]:
# Iniciar child run para esta tarea
start_child_run("03_register_embedding_model")

In [0]:
# Install required packages with specific versions for stability
%pip install mlflow torch transformers pandas==2.2.2 numpy==1.26.4
%restart_python

In [0]:
%run "./00 Configuration and Utils"

## Custom Embedding Model Class

This section defines a custom MLflow model class that wraps the embedding model for Databricks Vector Search compatibility.

In [0]:
import mlflow
import numpy as np
import pandas as pd
import torch
from typing import List
from transformers import AutoTokenizer, AutoModel

class SimpleEmbeddingModel(mlflow.pyfunc.PythonModel):
    """
    Custom embedding model for MLflow that is compatible with Databricks Vector Search.
    
    This class loads a transformers model, defines the logic for generating embeddings,
    and formats the output specifically as required by Databricks Vector Search.
    
    Key Features:
    - Batch processing for efficiency
    - Proper tensor handling and memory management
    - Vector Search compatible output format
    - Error handling and validation
    """
    
    def load_context(self, context):
        """
        This function executes once when the model is loaded into memory
        at the Model Serving endpoint. Ideal place for loading heavy models.
        """
        log_step("model_load", "started", f"Loading {EMBEDDING_MODEL_BASE}")
        
        try:
            # Load the tokenizer and pre-trained model from Hugging Face
            self.tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_BASE)
            self.model = AutoModel.from_pretrained(EMBEDDING_MODEL_BASE)
            self.max_length = MAX_LENGTH
            
            # Set model to evaluation mode
            self.model.eval()
            
            log_step("model_load", "success", f"Model loaded: {EMBEDDING_MODEL_BASE}")
            
        except Exception as e:
            log_step("model_load", "failed", f"Error loading model: {e}")
            raise e
    
    def encode(self, texts: List[str]) -> np.ndarray:
        """
        Helper function to process a list of texts and return their embeddings as a NumPy array.
        
        Args:
            texts: List of input texts to encode
            
        Returns:
            np.ndarray: Array of embeddings with shape (batch_size, embedding_dim)
        """
        try:
            # Tokenize the batch of texts with padding and truncation
            encoded_input = self.tokenizer(
                texts, 
                padding=True, 
                truncation=True, 
                max_length=self.max_length,
                return_tensors='pt'
            )
            
            # Inference without gradient calculation to save memory and time
            with torch.no_grad():
                model_output = self.model(**encoded_input)
                
            # Calculate embeddings using mean pooling of last_hidden_state
            # This is a simple and effective pooling strategy
            embeddings = model_output.last_hidden_state.mean(dim=1)
            
            # Return embeddings as NumPy array
            return embeddings.numpy()
            
        except Exception as e:
            log_step("encode_error", "failed", f"Error encoding texts: {e}")
            raise e

    def predict(self, context, model_input: pd.DataFrame) -> List[List[float]]:
        """
        Main method that MLflow calls for each inference request.
        Takes a DataFrame, extracts text, generates embeddings and returns
        output in the format required by Databricks Vector Search.
        
        Args:
            context: MLflow context (not used)
            model_input: DataFrame with 'input' column containing texts
            
        Returns:
            List[List[float]]: List of embeddings compatible with Vector Search
        """
        # --- 1. Input Validation and Processing ---
        if not isinstance(model_input, pd.DataFrame):
            raise TypeError(f"Input must be a pandas DataFrame, received {type(model_input)}")

        if "input" not in model_input.columns:
            raise ValueError("Input DataFrame must contain a column named 'input'")

        # Extract text column, fill null values, convert to string and then to Python list
        texts = model_input["input"].fillna('').astype(str).tolist()

        # --- 2. Embedding Generation ---
        try:
            embeddings = self.encode(texts).astype(np.float32)
            
            # Log processing info
            batch_size = len(texts)
            embedding_dim = embeddings.shape[1] if len(embeddings.shape) > 1 else 0
            
            if mlflow.active_run():
                mlflow.log_metric("batch_size", batch_size)
                mlflow.log_metric("embedding_dimension", embedding_dim)
            
        except Exception as e:
            log_step("prediction_error", "failed", f"Error generating embeddings: {e}")
            raise e

        # --- 3. Output Format for Vector Search ---
        # Return embeddings as list of lists of floats
        # This is the exact format that Databricks Vector Search expects
        return embeddings.tolist()

## Model Registration and MLflow Setup

This section registers the embedding model in MLflow with proper signatures and dependencies.

In [0]:
import mlflow
import mlflow.pyfunc
import torch
import transformers
import pandas as pd
from mlflow.models.signature import infer_signature

# Start MLflow run for model registration
mlflow.log_param("step", "embedding_model_registration")
mlflow.log_param("environment", ENVIRONMENT)
mlflow.log_param("base_model", EMBEDDING_MODEL_BASE)
mlflow.log_param("embedding_dimensions", EMBEDDING_DIM)
mlflow.log_param("max_length", MAX_LENGTH)
mlflow.log_param("framework", MODEL_FRAMEWORK)

log_step("model_registration", "started", f"Registering {EMBEDDING_MODEL_FULL}")

try:
    # Create input example for model signature
    input_example = pd.DataFrame({
        "input": [
            "Your string for the embedding model goes here",
            "This is another example text for testing",
            "Embedding models convert text to vectors"
        ]
    })

    # Instantiate the model and generate output for signature inference
    log_step("signature_creation", "started", "Creating model signature")
    
    model_instance = SimpleEmbeddingModel()
    model_instance.load_context(None)
    output_example = model_instance.predict(None, input_example)

    # Infer signature from input and output examples
    signature = infer_signature(input_example, output_example)
    print("Model signature inferred successfully:")
    print(signature)
    
    mlflow.log_param("input_schema", str(signature.inputs))
    mlflow.log_param("output_schema", str(signature.outputs))
    
    log_step("signature_creation", "success", "Model signature created")

    # Register the model with comprehensive metadata
    log_step("model_logging", "started", "Logging model to MLflow")
    
    model_info = mlflow.pyfunc.log_model(
        artifact_path="simple_embedding_model",
        python_model=SimpleEmbeddingModel(),
        registered_model_name=EMBEDDING_MODEL_FULL,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"torch=={torch.__version__}",
            f"transformers=={transformers.__version__}",
            "pandas==2.2.2",
            "numpy==1.26.4"
        ],
        metadata={
            "base_model": EMBEDDING_MODEL_BASE,
            "embedding_dimensions": EMBEDDING_DIM,
            "max_sequence_length": MAX_LENGTH,
            "framework": MODEL_FRAMEWORK,
            "compatible_with": "databricks_vector_search",
            "purpose": "text_embedding",
            "environment": ENVIRONMENT
        }
    )
    
    # Log model registration metrics
    mlflow.log_param("model_uri", model_info.model_uri)
    mlflow.log_param("model_version", model_info.registered_model_version)
    
    log_step("model_logging", "success", f"Model registered as version {model_info.registered_model_version}")
    
    print(f"✅ Model '{EMBEDDING_MODEL_FULL}' registered successfully!")
    print(f"📍 Model URI: {model_info.model_uri}")
    print(f"🔢 Version: {model_info.registered_model_version}")
    
except Exception as e:
    log_step("model_registration", "failed", f"Error registering model: {e}")
    raise e

In [0]:
# Test the registered model
log_step("model_testing", "started", "Testing registered model")

try:
    # Load the model for testing
    model_uri = f"models:/{EMBEDDING_MODEL_FULL}/latest"
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    
    # Test with sample data
    test_input = pd.DataFrame({
        "input": [
            "Databricks is a unified analytics platform",
            "MLflow helps manage the machine learning lifecycle"
        ]
    })
    
    # Generate embeddings
    test_output = loaded_model.predict(test_input)
    
    # Validate output format
    assert isinstance(test_output, list), "Output should be a list"
    assert len(test_output) == 2, "Should have 2 embeddings"
    assert isinstance(test_output[0], list), "Each embedding should be a list"
    assert len(test_output[0]) == EMBEDDING_DIM, f"Each embedding should have {EMBEDDING_DIM} dimensions"
    
    # Log test results
    mlflow.log_metric("test_batch_size", len(test_input))
    mlflow.log_metric("test_embedding_dim", len(test_output[0]))
    mlflow.log_param("test_status", "passed")
    
    log_step("model_testing", "success", f"Model test passed - output shape: {len(test_output)}x{len(test_output[0])}")
    
    print("🧪 Model Test Results:")
    print(f"   Input texts: {len(test_input)}")
    print(f"   Output embeddings: {len(test_output)}")
    print(f"   Embedding dimensions: {len(test_output[0])}")
    print("   ✅ All tests passed!")
    
except Exception as e:
    mlflow.log_param("test_status", "failed")
    log_step("model_testing", "failed", f"Model test failed: {e}")
    print(f"❌ Model test failed: {e}")

In [0]:
# Display sample embeddings for verification
print("📊 Sample Embedding Output:")
print("="*50)
if 'test_output' in locals():
    for i, embedding in enumerate(test_output[:2]):  # Show first 2 embeddings
        print(f"Text {i+1} embedding (first 10 dimensions):")
        print(f"  {embedding[:10]}...")
        print(f"  Total dimensions: {len(embedding)}")
        print()
else:
    print("No test output available")

In [0]:
# Finalizar child run
try:
    log_step("embedding_model_registration", "completed", "Embedding model registration finished")
    end_child_run("success")
    print("✅ Child run finalizada correctamente")
except Exception as e:
    print(f"⚠️ Error finalizando child run: {e}")
    end_child_run("failed")